# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, getpass

# Token order: environment variable -> Colab Secret -> prompt
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connection ready.")
print("Available table sources:")
for name in TABLES:
    print("-", name)

DuckDB connection ready.
Available table sources:
- dim_clients
- dim_content
- fact_daily
- fact_daily_sample
- fact_query_90d


In [4]:
# Build the earlier training snapshot
TRAIN_END = "2026-04-30"

train_snapshot = con.sql(f"""
    WITH windowed AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 60 DAY
                     AND report_date <= DATE '{TRAIN_END}' - INTERVAL 30 DAY
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']}

        WHERE report_date <= DATE '{TRAIN_END}'
          AND report_date > DATE '{TRAIN_END}' - INTERVAL 60 DAY

        GROUP BY client_hash_id, content_hash_id
    )

    SELECT *
    FROM windowed
    WHERE imp_prev30 >= 100
""").df()

train_snapshot["is_declining"] = (
    train_snapshot["imp_last30"]
    < 0.8 * train_snapshot["imp_prev30"]
).astype(int)

print("Train snapshot:", train_snapshot.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train snapshot: (100849, 7)


In [5]:
# Build the June 30 test snapshot
TEST_END = "2026-06-30"

test_snapshot = con.sql(f"""
    WITH windowed AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 60 DAY
                     AND report_date <= DATE '{TEST_END}' - INTERVAL 30 DAY
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']}

        WHERE report_date <= DATE '{TEST_END}'
          AND report_date > DATE '{TEST_END}' - INTERVAL 60 DAY

        GROUP BY client_hash_id, content_hash_id
    )

    SELECT *
    FROM windowed
    WHERE imp_prev30 >= 100
""").df()

test_snapshot["is_declining"] = (
    test_snapshot["imp_last30"]
    < 0.8 * test_snapshot["imp_prev30"]
).astype(int)

print("Test snapshot:", test_snapshot.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test snapshot: (111247, 7)


In [6]:
from sklearn.ensemble import RandomForestClassifier

features_w06 = [
    "imp_prev30",
    "clk_last30",
    "pos_last30"
]

X_train = train_snapshot[features_w06]
y_train = train_snapshot["is_declining"]

X_test = test_snapshot[features_w06]
y_test = test_snapshot["is_declining"]

model_w06 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_w06.fit(X_train, y_train)

y_pred = model_w06.predict(X_test)
y_prob = model_w06.predict_proba(X_test)[:, 1]

print("Model and test predictions ready.")
print("X_test:", X_test.shape)
print("Predictions:", len(y_pred))
print("Probabilities:", len(y_prob))

Model and test predictions ready.
X_test: (111247, 3)
Predictions: 111247
Probabilities: 111247


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue ranks test-snapshot records by the model's predicted probability
of `is_declining = 1`. Records with a higher predicted probability are placed
earlier in the review queue.

This ranking is a decision-support signal, not a confirmed diagnosis of decline.
The time-aware validation measured a ROC-AUC of 0.709 and an accuracy of 60.85%.
The test positive rate was 64.31%. These results do not support treating the
model as a reliable standalone predictor.

The queue uses the three model features:

- `imp_prev30`
- `clk_last30`
- `pos_last30`

Reason codes are simple review signals based on the observed feature values:

- `REVIEW_VISIBILITY` — the prior-period impression signal is relatively low.
- `REVIEW_ENGAGEMENT` — the recent click signal is relatively low.
- `REVIEW_POSITION` — the position signal is relatively weak when a position
  value is available.
- `MULTIPLE_REVIEW_SIGNALS` — more than one review signal is present.
- `GENERAL_REVIEW` — no specific review signal was assigned.

These reason codes are not causal explanations. A low value in a feature does
not establish that the feature caused a decline.

The queue is intended to help a human reviewer decide what to inspect first.
No content change should be made automatically from the ranking alone.

In [10]:
import pandas as pd

# Build the action queue from the time-aware test set.
action_queue = X_test.copy().reset_index(drop=True)

action_queue["decline_probability"] = y_prob
action_queue["predicted_decline"] = y_pred

# Descriptive markers based on the test-set distributions.
imp_median = action_queue["imp_prev30"].median()
clk_median = action_queue["clk_last30"].median()
pos_median = action_queue["pos_last30"].median()


def get_reason_codes(row):
    reasons = []

    if pd.notna(row["imp_prev30"]) and row["imp_prev30"] <= imp_median:
        reasons.append("REVIEW_VISIBILITY")

    if pd.notna(row["clk_last30"]) and row["clk_last30"] <= clk_median:
        reasons.append("REVIEW_ENGAGEMENT")

    if pd.notna(row["pos_last30"]) and row["pos_last30"] >= pos_median:
        reasons.append("REVIEW_POSITION")

    if len(reasons) >= 2:
        reasons.append("MULTIPLE_REVIEW_SIGNALS")

    if not reasons:
        reasons.append("GENERAL_REVIEW")

    return reasons


action_queue["reason_codes"] = action_queue.apply(
    get_reason_codes,
    axis=1
)

# Rank by model score.
# This is a review-priority ranking, not an automatic decision.
action_queue = action_queue.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = range(1, len(action_queue) + 1)

print("Action queue shape:", action_queue.shape)
action_queue.head(10)

Action queue shape: (111247, 7)


,imp_prev30,clk_last30,pos_last30,decline_probability,predicted_decline,reason_codes,rank
0,275.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",1
1,115.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",2
2,328.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",3
3,107.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",4
4,241.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",5
5,174.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",6
6,110.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",7
7,113.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",8
8,223.0,0.0,NaN,1.0,1,"[REVIEW_VISIBILITY, REVIEW_ENGAGEMENT, MULTIPL...",9
9,657.0,0.0,73.605009,1.0,1,"[REVIEW_ENGAGEMENT, REVIEW_POSITION, MULTIPLE_...",10


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for content and SEO reviewers who need to prioritize
which records to inspect first.

The model output can be used as a decision-support signal for review
prioritization. A higher predicted probability means that the record is placed
earlier in the review queue; it does not mean that decline has been confirmed.

The playbook is limited to the signals available in the validated model:
`imp_prev30`, `clk_last30`, and `pos_last30`. It does not include other
business, content, technical, or search-context information that a human
reviewer may have.

The model should not be used to automatically edit, remove, publish, or
otherwise change content. It should also not be treated as a causal model,
a guarantee of future performance, or a production decision system.

The recommendations may become less useful when the underlying data
distribution or content environment changes. Human review remains required
before any action is taken.

In [11]:
# Basic checks for intended use and model limits

print("Records in review queue:", len(action_queue))
print("Model features:", features_w06)
print("Missing values by feature:")
print(action_queue[features_w06].isna().sum())

print("\nPredicted decline rate:",
      action_queue["predicted_decline"].mean())

print("Observed test positive rate:",
      y_test.mean())

Records in review queue: 111247
Model features: ['imp_prev30', 'clk_last30', 'pos_last30']
Missing values by feature:
imp_prev30       0
clk_last30       0
pos_last30    1160
dtype: int64

Predicted decline rate: 0.43903206378598975
Observed test positive rate: 0.6431364441288304


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human reviewer must check a record before any content action is taken.

The reviewer should check:

- whether the model-ranked signal is consistent with the available performance data;
- whether missing values or unusual values affect the review;
- whether there is relevant context not represented by the three model features;
- whether the observed change appears meaningful rather than a short-term fluctuation;
- whether the proposed content action is appropriate for the specific record.

The model output should not be used to automatically:

- delete or unpublish content;
- rewrite or publish content;
- change important content or SEO settings without review;
- make claims about why a record declined;
- make decisions about clients or content without additional context;
- treat a high model probability as proof of decline.

The playbook is a review-prioritization tool. Final content decisions remain with a human reviewer.

In [12]:
# Human-review checks for the action queue

review_checks = pd.DataFrame({
    "check": [
        "Total records in queue",
        "Records with missing position",
        "Records with a predicted decline",
        "Records without a predicted decline",
    ],
    "count": [
        len(action_queue),
        action_queue["pos_last30"].isna().sum(),
        (action_queue["predicted_decline"] == 1).sum(),
        (action_queue["predicted_decline"] == 0).sum(),
    ]
})

review_checks["share"] = (
    review_checks["count"] / len(action_queue)
)

review_checks

,check,count,share
0,Total records in queue,111247,1.000000
1,Records with missing position,1160,0.010427
2,Records with a predicted decline,48841,0.439032
3,Records without a predicted decline,62406,0.560968


In [13]:
# Confirm that no record is automatically assigned a content action.
action_queue["action"] = "HUMAN_REVIEW_REQUIRED"

print("Automatic content actions:",
      (action_queue["action"] != "HUMAN_REVIEW_REQUIRED").sum())

print("Human-review records:",
      (action_queue["action"] == "HUMAN_REVIEW_REQUIRED").sum())

Automatic content actions: 0
Human-review records: 111247


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored for signs that the model or its review
signals are becoming stale.

The main triggers are:

- a meaningful change in the distribution of the three model features;
- a substantial increase in missing values, especially `pos_last30`;
- a noticeable change in the share of records receiving a high review priority;
- a drop in validation performance when a later time-aware test snapshot becomes available;
- changes in the data or content environment that are not represented in the current features.

A later time-aware validation should be used before deciding that the model needs
to be retrained.

Retraining should not be triggered by a single unusual observation. A persistent
change in the data distribution or a sustained deterioration in validation
performance should be investigated first.

The monitoring checks are intended as lightweight research-stage triggers, not
as production monitoring guarantees.

In [14]:
# Lightweight monitoring checks for the current review queue

monitoring_summary = pd.DataFrame({
    "metric": [
        "queue_records",
        "predicted_decline_share",
        "missing_imp_prev30_share",
        "missing_clk_last30_share",
        "missing_pos_last30_share",
    ],
    "value": [
        len(action_queue),
        action_queue["predicted_decline"].mean(),
        action_queue["imp_prev30"].isna().mean(),
        action_queue["clk_last30"].isna().mean(),
        action_queue["pos_last30"].isna().mean(),
    ]
})

print("Monitoring summary:")
display(monitoring_summary)


# Current feature distributions for future comparison.
feature_summary = action_queue[features_w06].describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
]

print("\nCurrent feature distribution:")
display(feature_summary)

Monitoring summary:


,metric,value
0,queue_records,111247.000000
1,predicted_decline_share,0.439032
2,missing_imp_prev30_share,0.000000
3,missing_clk_last30_share,0.000000
4,missing_pos_last30_share,0.010427



Current feature distribution:


,count,mean,std,min,50%,max
imp_prev30,111247.0,2290.660665,6942.053859,100.0,565.000000,585502.0
clk_last30,111247.0,6.972350,32.517226,0.0,1.000000,3817.0
pos_last30,110087.0,20.518548,17.892419,0.0,13.604974,579.0


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked review queue is exported to `work/outputs/` so that the paper can
reuse the same decision-support output without manually rebuilding the ranking.

The exported queue contains the model score, prediction, review signals,
reason codes, and review rank.

The queue is generated from the test snapshot and is kept outside version
control because it contains derived data records.

Figures that are useful for the paper can be saved separately under
`work/figures/`. Only non-sensitive, reusable research artifacts should be
committed.

In [15]:
import os

# Create the output directory.
os.makedirs("work/outputs", exist_ok=True)

# Export the ranked review queue.
queue_path = "work/outputs/action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

print("Exported:", queue_path)
print("Rows:", len(action_queue))
print("Columns:", list(action_queue.columns))

Exported: work/outputs/action_queue.csv
Rows: 111247
Columns: ['imp_prev30', 'clk_last30', 'pos_last30', 'decline_probability', 'predicted_decline', 'reason_codes', 'rank', 'action']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [16]:
# Final self-check

checks = {
    "queue_created": len(action_queue) > 0,
    "rank_complete": action_queue["rank"].is_monotonic_increasing,
    "no_automatic_actions": (
        action_queue["action"] == "HUMAN_REVIEW_REQUIRED"
    ).all(),
    "export_exists": os.path.exists("work/outputs/action_queue.csv"),
    "all_required_features_present": all(
        feature in action_queue.columns for feature in features_w06
    ),
    "reason_codes_present": action_queue["reason_codes"].notna().all(),
}

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

queue_created: PASS
rank_complete: PASS
no_automatic_actions: PASS
export_exists: PASS
all_required_features_present: PASS
reason_codes_present: PASS
